In [2]:
import os
import pandas as pd
from pathlib import Path
import sys
import numpy as np
import re


df_train = pd.read_json("dataset/bigvul_train.jsonl", lines=True)
df_val = pd.read_json("dataset/bigvul_val.jsonl", lines=True) 
df_test = pd.read_json("dataset/bigvul_test.jsonl", lines=True) 
df = pd.concat([df_train, df_val, df_test], ignore_index=True)

# 向上两级到项目根目录
PROJECT_ROOT = Path(os.path.abspath("")).resolve()
while not (PROJECT_ROOT / "configs").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

from src.utils.cwe import CWEUtils
from src.utils.cve import CVEUtils

MIN_CODE_LEN = 35  # 代码最短字符数（过滤过短的无意义代码）
MAX_CODE_LEN = 65535   # 代码最长字符数（过滤过长的异常代码）

print(len(df))
df.head()

Project root: <PROJECT_ROOT>
188636


,CWE ID,CVE ID,func,target,idx,flaw_line,flaw_line_index
0,CWE-20,CVE-2018-6111,BrowserContext* SharedWorkerDevToolsAgentHost:...,1,186765,RenderProcessHost* rph = GetProcess();,1
1,CWE-264,CVE-2016-3890,"asocket* create_remote_socket(unsigned id, atr...",0,171392,NaN,NaN
2,CWE-119,CVE-2016-10066,static void draw_under_color_string(DrawingWan...,0,71810,NaN,NaN
3,CWE-79,CVE-2016-5164,static bool isCommandLineAPIGetter(const Strin...,0,141681,NaN,NaN
4,NaN,CVE-2016-3751,static void rpng2_x_display_row(ulg row)\n{\n ...,0,172977,NaN,NaN


In [3]:
# 只保留有漏洞的样本
df = df[df['target'] == 1] 


df = df[
    df['CWE ID'].notna()
    & (df['CWE ID'].str.strip() != '')
    & (df['CWE ID'].str.strip() != 'None')
    & (df['CVE ID'].notna())
    & (df['CVE ID'].str.strip() != '')
    & (df['CVE ID'].str.strip() != 'None')
    & (df['func'].notna())
    & (df['func'].str.strip() != '')
    & (df['func'].str.strip() != 'None')
    #& df['flaw_line'].notna()
    #& (df['flaw_line'].str.strip() != '')
]

df = df.rename(columns={
    'CWE ID': 'cwe_id',
    'CVE ID': 'cve_id',
    'idx': 'id',
    'func': 'vul_code'
})

# 移除'target'列、'flaw_line'列和'flaw_line_index'列
df = df.drop(columns=['target', 'flaw_line', 'flaw_line_index'])

# 过滤掉 vul_code 过短(<MIN_CODE_LEN) 或过长(>MAX_CODE_LEN) 的行
vul_len = df['vul_code'].str.len()
df = df[vul_len.between(MIN_CODE_LEN, MAX_CODE_LEN, inclusive='both')]

# 新增一个'source'列
df['source'] = 'BigVul'

print(len(df))
df.head()

8763


,cwe_id,cve_id,vul_code,id,source
0,CWE-20,CVE-2018-6111,BrowserContext* SharedWorkerDevToolsAgentHost:...,186765,BigVul
54,CWE-119,CVE-2007-6761,static int __videobuf_mmap_mapper(struct video...,182089,BigVul
58,CWE-399,CVE-2011-2880,void CloudPolicyController::SetState(\nCloudPo...,183738,BigVul
106,CWE-264,CVE-2011-2795,void HTMLElement::setOuterHTML(const String& h...,183899,BigVul
111,CWE-19,CVE-2015-8952,static int __init init_ext2_fs(void)\n{\n\tint...,183147,BigVul


In [ ]:
# 新增并填充 'cwe_desc' 列
cwe_utils = CWEUtils(PROJECT_ROOT / "data/raw/cwe/cwec_v4.19.1.xml")

def get_cwe_desc(cwe_field):
    """根据 cwe 字段生成 CWE prompt 文本（使用 CWEUtils.get_prompt）。

    支持多种 cwe 格式：numpy数组、列表、字符串等。
    返回值为结构化的 prompt 文本；若无法解析/查询则返回 None。
    """
    if cwe_field is None:
        return None

    cwe_id = None

    # 处理 numpy 数组格式（如 array(['CWE-20'], dtype=object)）
    if isinstance(cwe_field, np.ndarray) and len(cwe_field) > 0:
        cwe_id = cwe_field[0]
    # 处理列表格式，如 ["CWE-20"] 或 ["CWE-20", "CWE-79"]
    elif isinstance(cwe_field, list) and len(cwe_field) > 0:
        cwe_id = cwe_field[0]
    # 处理字符串格式
    elif isinstance(cwe_field, str):
        match = re.search(r"CWE-\d+", cwe_field)
        if match:
            cwe_id = match.group()
        else:
            cwe_id = cwe_field.strip().strip("[]'\"")
            if "," in cwe_id:
                cwe_id = cwe_id.split(",")[0].strip().strip("'\"")

    if cwe_id is None or str(cwe_id).strip() == "":
        return None

    prompt = cwe_utils.get_prompt(str(cwe_id).strip())
    if "Result: The requested entry was not found in the CWE dataset." in prompt:
        return None

    return prompt

# 填充 cwe_desc 列
df['cwe_desc'] = df['cwe_id'].apply(get_cwe_desc)

# 验证结果
print(f"总样本数: {len(df)}")
print(f"cwe_desc 为空的样本数: {df['cwe_desc'].isna().sum()}")
print(f"cwe_desc 填充率: {(1 - df['cwe_desc'].isna().sum() / len(df)) * 100:.2f}%")
df.head()


总样本数: 8763
cwe_desc 为空的样本数: 0
cwe_desc 填充率: 100.00%


,cwe_id,cve_id,vul_code,id,source,cwe_desc
0,CWE-20,CVE-2018-6111,BrowserContext* SharedWorkerDevToolsAgentHost:...,186765,BigVul,- CWE ID: CWE-20\n- Name: Improper Input Valid...
54,CWE-119,CVE-2007-6761,static int __videobuf_mmap_mapper(struct video...,182089,BigVul,- CWE ID: CWE-119\n- Name: Improper Restrictio...
58,CWE-399,CVE-2011-2880,void CloudPolicyController::SetState(\nCloudPo...,183738,BigVul,- CWE ID: CWE-399\n- Name: Resource Management...
106,CWE-264,CVE-2011-2795,void HTMLElement::setOuterHTML(const String& h...,183899,BigVul,"- CWE ID: CWE-264\n- Name: Permissions, Privil..."
111,CWE-19,CVE-2015-8952,static int __init init_ext2_fs(void)\n{\n\tint...,183147,BigVul,- CWE ID: CWE-19\n- Name: Data Processing Erro...


In [5]:
# 新增并填充 'cve_desc' 列
cve_utils = CVEUtils(PROJECT_ROOT / "data/raw/cwe/CVE_v2026.03.18.json")

def get_cve_desc(cve_field):
    """根据 cwe 字段生成 CWE prompt 文本（使用 CWEUtils.get_prompt）。

    支持多种 cwe 格式：numpy数组、列表、字符串等。
    返回值为结构化的 prompt 文本；若无法解析/查询则返回 None。
    """
    if cve_field is None:
        return None

    cve_id = None

    # 处理 numpy 数组格式（如 array(['CWE-20'], dtype=object)）
    if isinstance(cve_field, np.ndarray) and len(cve_field) > 0:
        cve_id = cve_field[0]
    # 处理列表格式，如 ["CWE-20"] 或 ["CWE-20", "CWE-79"]
    elif isinstance(cve_field, list) and len(cve_field) > 0:
        cve_id = cve_field[0]
    # 处理字符串格式
    elif isinstance(cve_field, str):
        match = re.search(r"CWE-\d+", cve_field)
        if match:
            cve_id = match.group()
        else:
            cve_id = cve_field.strip().strip("[]'\"")
            if "," in cve_id:
                cve_id = cve_id.split(",")[0].strip().strip("'\"")

    if cve_id is None or str(cve_id).strip() == "":
        return None

    prompt = cve_utils.get_prompt(str(cve_id).strip())
    if "Result: The requested entry was not found in the CWE dataset." in prompt:
        return None

    return prompt

# 填充 cve_desc 列
df['cve_desc'] = df['cve_id'].apply(get_cve_desc)

# 验证结果
print(f"总样本数: {len(df)}")
print(f"cwe_desc 为空的样本数: {df['cwe_desc'].isna().sum()}")
print(f"cwe_desc 填充率: {(1 - df['cwe_desc'].isna().sum() / len(df)) * 100:.2f}%")
df.head()


总样本数: 8763
cwe_desc 为空的样本数: 0
cwe_desc 填充率: 100.00%


,cwe_id,cve_id,vul_code,id,source,cwe_desc,cve_desc
0,CWE-20,CVE-2018-6111,BrowserContext* SharedWorkerDevToolsAgentHost:...,186765,BigVul,- CWE ID: CWE-20\n- Name: Improper Input Valid...,- CVE ID: CVE-2018-6111\n- Status: Modified\n-...
54,CWE-119,CVE-2007-6761,static int __videobuf_mmap_mapper(struct video...,182089,BigVul,- CWE ID: CWE-119\n- Name: Improper Restrictio...,- CVE ID: CVE-2007-6761\n- Status: Deferred\n-...
58,CWE-399,CVE-2011-2880,void CloudPolicyController::SetState(\nCloudPo...,183738,BigVul,- CWE ID: CWE-399\n- Name: Resource Management...,- CVE ID: CVE-2011-2880\n- Status: Deferred\n-...
106,CWE-264,CVE-2011-2795,void HTMLElement::setOuterHTML(const String& h...,183899,BigVul,"- CWE ID: CWE-264\n- Name: Permissions, Privil...",- CVE ID: CVE-2011-2795\n- Status: Deferred\n-...
111,CWE-19,CVE-2015-8952,static int __init init_ext2_fs(void)\n{\n\tint...,183147,BigVul,- CWE ID: CWE-19\n- Name: Data Processing Erro...,- CVE ID: CVE-2015-8952\n- Status: Deferred\n-...


In [6]:
# 将数据以jsonl的格式保存到data/raw/vul/Primevul.jsonl
output_path = PROJECT_ROOT / "data/raw/vul/BigVul.jsonl"
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_json(
    output_path,
    orient="records",
    lines=True,
    force_ascii=False
)

print(f"保存完成: {output_path} （共 {len(df)} 条样本）")

保存完成: <PROJECT_ROOT>/data/raw/vinj/BigVul.jsonl （共 8763 条样本）
